# Bundestagswahl 2025: Wählerwanderung

This Sankey compares labeled vote shares for the 2021 and 2025 Bundestag elections with selected estimated voter-migration flows between them. Node labels show the official share; link widths encode absolute estimated voters.

> **Method note:** The flows are model-based estimates from tagesschau / infratest dimap graphics, not exact individual ballot tracking or a complete balanced migration matrix. Party shares use valid second votes, while non-voter shares use eligible voters. Wahlrecht.de provides election-result context, not the complete voter-migration matrix.

The chart is built from `data/bundestagswahl_2025_voter_migration_sankey_plot_ready.csv`. The export cell writes the responsive HTML page to `p00/output`.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


def find_project_root(start: Path = Path.cwd()) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if candidate.name == "p00":
            return candidate
    raise RuntimeError(f"Could not find a parent folder named 'p00' from {start.resolve()}")


project_root = find_project_root()
csv_path = project_root / "data" / "bundestagswahl_2025_voter_migration_sankey_plot_ready.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Migration CSV not found: {csv_path}")

data = pd.read_csv(csv_path, encoding="utf-8")
required_columns = {
    "record_type",
    "source_id",
    "source_label",
    "source_party_long",
    "source_year",
    "source_vote_share_pct",
    "source_share_basis",
    "source_votes_total",
    "source_sort_order",
    "source_x",
    "source_color_hex",
    "target_id",
    "target_label",
    "target_party_long",
    "target_year",
    "target_vote_share_pct",
    "target_share_basis",
    "target_votes_total",
    "target_sort_order",
    "target_x",
    "target_color_hex",
    "value_voters",
    "flow_label",
    "flow_color_rgba",
    "estimate_type",
    "migration_source_url",
    "plot_note",
}
missing_columns = required_columns - set(data.columns)
if missing_columns:
    raise ValueError(f"Migration CSV is missing required columns: {sorted(missing_columns)}")

data = data.loc[data["record_type"].eq("flow")].copy()
data = data.dropna(subset=["source_id", "target_id", "source_label", "target_label", "value_voters"])
data["value_voters"] = pd.to_numeric(data["value_voters"], errors="raise")
if data.empty or (data["value_voters"] <= 0).any():
    raise ValueError("Migration CSV must contain positive voter estimates.")
if set(data["source_year"].unique()) != {2021} or set(data["target_year"].unique()) != {2025}:
    raise ValueError("Expected migration flows from 2021 sources to 2025 targets.")

print(f"Loaded {len(data)} estimated flows ({data['value_voters'].sum():,.0f} voters represented).")

In [ ]:
share_basis_labels = {
    "valid_second_votes": "Valid second votes",
    "eligible_voters": "Eligible voters",
}


def build_nodes(side: str) -> pd.DataFrame:
    columns = {
        f"{side}_id": "node_id",
        f"{side}_label": "label",
        f"{side}_party_long": "party_long",
        f"{side}_year": "year",
        f"{side}_vote_share_pct": "vote_share_pct",
        f"{side}_share_basis": "share_basis",
        f"{side}_votes_total": "votes_total",
        f"{side}_sort_order": "sort_order",
        f"{side}_x": "x",
        f"{side}_color_hex": "color_hex",
    }
    nodes = data[list(columns)].drop_duplicates(subset=f"{side}_id").rename(columns=columns)
    nodes = nodes.sort_values("sort_order").reset_index(drop=True)
    nodes["y"] = [(index + 0.5) / len(nodes) for index in range(len(nodes))]
    return nodes


def visible_node_color(color_hex: str) -> str:
    if color_hex.lower() in {"#000000", "#111111"}:
        return "#D4D8DE"
    return color_hex


def rgba_from_hex(color_hex: str, alpha: float = 0.38) -> str:
    color_hex = color_hex.lstrip("#")
    red, green, blue = (int(color_hex[index:index + 2], 16) for index in (0, 2, 4))
    return f"rgba({red},{green},{blue},{alpha})"


source_nodes = build_nodes("source")
target_nodes = build_nodes("target")
nodes = pd.concat([source_nodes, target_nodes], ignore_index=True)
node_index = {node_id: index for index, node_id in enumerate(nodes["node_id"])}
node_colors = [visible_node_color(color) for color in nodes["color_hex"]]
node_customdata = [
    [
        row.party_long,
        f"{int(row.votes_total):,}",
        share_basis_labels.get(row.share_basis, row.share_basis),
    ]
    for row in nodes.itertuples(index=False)
]

link_colors = data["flow_color_rgba"].tolist()
for index, color_hex in enumerate(data["flow_color_hex"]):
    if color_hex.lower() in {"#000000", "#111111"}:
        link_colors[index] = rgba_from_hex("#D4D8DE")
link_customdata = data[["flow_label", "estimate_type", "migration_source_url"]].values.tolist()

fig = go.Figure(
    go.Sankey(
        arrangement="freeform",
        node=dict(
            label=nodes["label"].tolist(),
            x=nodes["x"].astype(float).tolist(),
            y=nodes["y"].tolist(),
            color=node_colors,
            customdata=node_customdata,
            pad=22,
            thickness=18,
            line=dict(color="rgba(245,247,250,0.35)", width=0.7),
            hovertemplate=(
                "%{label}<br>%{customdata[0]}<br>%{customdata[1]} total votes"
                "<br>Share basis: %{customdata[2]}<extra></extra>"
            ),
        ),
        link=dict(
            source=[node_index[node_id] for node_id in data["source_id"]],
            target=[node_index[node_id] for node_id in data["target_id"]],
            value=data["value_voters"].astype(int).tolist(),
            color=link_colors,
            customdata=link_customdata,
            hovertemplate=(
                "%{customdata[0]}<br>%{value:,.0f} estimated voters"
                "<br>%{customdata[1]}<br>Source: %{customdata[2]}<extra></extra>"
            ),
        ),
    )
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(family="system-ui, sans-serif", size=13, color="#f5f7fa"),
    width=None,
    height=820,
    autosize=True,
    margin=dict(l=18, r=18, t=50, b=22),
    annotations=[
        dict(x=0.02, y=1.04, xref="paper", yref="paper", text="<b>2021 VOTE CHOICE</b>", showarrow=False, xanchor="left"),
        dict(x=0.98, y=1.04, xref="paper", yref="paper", text="<b>2025 VOTE CHOICE</b>", showarrow=False, xanchor="right"),
    ],
)
fig.show()

In [ ]:
from html import escape

project_root = find_project_root()
output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "20261008_bundestagswahl_sankey.html"

migration_urls = data["migration_source_url"].dropna().astype(str).drop_duplicates().tolist()
migration_url = next((url for url in migration_urls if "/charts/" not in url), migration_urls[0])
election_url_2021 = data["election_source_url_2021"].dropna().astype(str).iloc[0]
election_url_2025 = data["election_source_url_2025"].dropna().astype(str).iloc[0]
wahlrecht_url = data["wahlrecht_summary_url"].dropna().astype(str).iloc[0]
plot_note = escape(str(data["plot_note"].dropna().iloc[0]))

plot_html = fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    config={"responsive": True, "displaylogo": False},
)

html_document = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <meta name="theme-color" content="#0d1117">
  <title>Bundestag voter migration 2021–2025 | phey.app</title>
  <style>
    * {{ box-sizing: border-box; }}
    html, body {{ margin: 0; min-height: 100%; background: #0d1117; color: #f5f7fa; }}
    body {{ font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }}
    .plot-page {{ min-height: 100vh; padding: clamp(0.75rem, 2vw, 1.5rem); }}
    .plot-wrap {{ width: 100%; margin: 0 auto; }}
    .back-link {{ display: inline-flex; margin-bottom: 1rem; color: #c7d0dd; text-decoration: none; }}
    .back-link:hover {{ color: #ffffff; }}
    h1 {{ margin: 0; font-size: clamp(1.55rem, 3vw, 2.25rem); font-weight: 650; }}
    .subtitle {{ max-width: 75rem; margin: 0.45rem 0 0; color: #c7d0dd; line-height: 1.5; }}
    .method-note {{ max-width: 90rem; margin: 0.75rem 0 0.25rem; padding-left: 0.8rem; border-left: 3px solid #d4a72c; color: #aeb8c4; line-height: 1.5; }}
    .sources {{ margin: 0.75rem 0 1.5rem; color: #8b949e; font-size: 0.9rem; line-height: 1.7; }}
    .sources a {{ color: #8ecae6; }}
    @media (max-width: 640px) {{ .method-note {{ font-size: 0.92rem; }} }}
  </style>
</head>
<body>
  <main class="plot-page">
    <div class="plot-wrap">
      <a class="back-link" href="../index.html">← Back to phey.app</a>
      <header>
        <h1>Bundestag voter migration, 2021 to 2025</h1>
        <p class="subtitle">Flows show estimated voter migration in absolute voters. Party labels show vote shares for the respective Bundestag election.</p>
        <p class="method-note"><strong>Method:</strong> These are model-based estimates, not exact ballot tracking. {plot_note} Wahlrecht.de is used for election-results context and does not provide the complete voter-migration matrix.</p>
        <p class="sources">Sources: <a href="{escape(migration_url, quote=True)}" target="_blank" rel="noopener noreferrer">tagesschau / infratest dimap migration estimates</a>; official <a href="{escape(election_url_2021, quote=True)}" target="_blank" rel="noopener noreferrer">2021</a> and <a href="{escape(election_url_2025, quote=True)}" target="_blank" rel="noopener noreferrer">2025 election results</a>; <a href="{escape(wahlrecht_url, quote=True)}" target="_blank" rel="noopener noreferrer">Wahlrecht.de</a> for context.</p>
      </header>
      {plot_html}
    </div>
  </main>
</body>
</html>
"""

output_file.write_text(html_document, encoding="utf-8")
print(f"Exported Bundestag Sankey to: {output_file.resolve()}")